# 11 — The final model, and predicting a fight

After the test look, the model is refitted on **every decided fight from 2014 to the last
event in the data**, with the same frozen recipe. This is the model that predicts upcoming
fights. Its own score cannot be measured (no unseen fights are left); the test look in
`10_test_evaluation.ipynb` is the honest estimate of how well this recipe predicts.

The calibration stretch is the one fitted in the test look: refitting it here would need
out-of-fold predictions on the test years, which the tuning guard refuses on purpose.

`modeling.final.predict_upcoming` adds a fight to the history as a fight with no statistics
and no result, then reads its features from the normal pipeline: the same code that built
every past row, so nothing about a future fight is computed differently.

In [1]:
import json
import pandas as pd

from ufc_predictor.config import RAW_DIR
from ufc_predictor.data.io import load_fighters, load_processed
from ufc_predictor.features.elo import load_elo_params, load_elo_weights
from ufc_predictor.modeling.final import fit_final, load_bundle, predict_upcoming, save_bundle
from ufc_predictor.modeling.tune import load_params

fights, fighters = load_processed("fights"), load_fighters()
source = json.loads((RAW_DIR / "source" / "SOURCE.json").read_text())
until = pd.Timestamp(source["last_event"]) + pd.Timedelta(days=1)
bundle = fit_final(fights, fighters, until=until, params=load_params(),
                   elo_k=load_elo_params(), elo_weights=load_elo_weights(), calibrate=False)
bundle["stretch"] = json.load(open("reviews/test_result_2026-10-17.json"))["stretch"]
bundle["data_version"] = source["version"]
path = save_bundle(bundle)
print(f"trained on {bundle['training_fights']:,} fights to {source['last_event']} (data v{source['version']}); "
      f"stretch {bundle['stretch']:.3f}; saved to {path.name}")

trained on 6,276 fights to 2026-10-10 (data v20); stretch 1.235; saved to final_model.joblib


## Predicting a fight

An example, **not a scheduled bout**: rematches of two fights from the last card in the data,
as if held on 2026-12-05. Fighters are named as in the fighter table (or by id); the
probability is for the fighter listed first.

In [2]:
bouts = [
    {"fighter_a": "Brendan Allen", "fighter_b": "Christian Leroy Duncan", "date": "2026-12-05", "weight_class": "Middleweight Bout"},
    {"fighter_a": "Andre Fili", "fighter_b": "Kai Kamaka III", "date": "2026-12-05", "weight_class": "Featherweight Bout"},
]
predict_upcoming(bouts, load_bundle())

,date,fighter,opponent,p_win,fighter_debut,opponent_debut
0,2026-12-05,Brendan Allen,Christian Leroy Duncan,0.542798,False,False
1,2026-12-05,Andre Fili,Kai Kamaka III,0.563904,False,False


## Pound for pound, by Elo

Every active fighter ranked by their Elo after their last fight, minus a penalty for inactivity:
nothing for the first 8 months without a fight, then 0.5 x (months - 8)^2 points (8 at 12 months, 50
at 18, 128 at 24), the size the comeback data supports. After 18 months without a fight a fighter is
not ranked. At least 3 UFC fights; men and women apart, since they never meet. Rebuilt every time
this notebook runs, so it follows each data refresh. The model's own Elo input is unchanged.

In [3]:
from ufc_predictor.config import PROCESSED_DIR
from ufc_predictor.modeling.present import p4p

ranking = p4p(bundle["featured"], bundle["elo_k"], bundle["elo_weights"])
for label, table in ranking.items():
    table.to_csv(PROCESSED_DIR / f"p4p_{label}.csv", index=False)
print(f"as of {source['last_event']} (data v{source['version']})")
ranking["men"]

as of 2026-10-10 (data v20)


,rank,name,rating,elo,penalty,months_inactive,division,record,last_fight
0,1,Islam Makhachev,2185,2185,0,1.8,welterweight,18-1,2026-08-15
1,2,Ciryl Gane,2089,2089,0,3.9,heavyweight,11-2,2026-06-14
2,3,Carlos Ulberg,2046,2046,0,6.0,light_heavyweight,10-1,2026-04-11
3,4,Justin Gaethje,2045,2045,0,3.9,lightweight,11-5,2026-06-14
4,5,Carlos Prates,2043,2043,0,5.3,welterweight,7-1,2026-05-02
5,6,Tom Aspinall,2038,2044,6,11.5,heavyweight,8-1,2025-10-25
6,7,Charles Oliveira,2027,2027,0,7.1,lightweight,25-11,2026-03-07
7,8,Khamzat Chimaev,2023,2023,0,5.1,middleweight,9-1,2026-05-09
8,9,Arman Tsarukyan,2018,2018,0,0.7,lightweight,11-2,2026-09-19
9,10,Dricus Du Plessis,2018,2018,0,2.8,middleweight,10-1,2026-07-18


In [4]:
ranking["women"]

,rank,name,rating,elo,penalty,months_inactive,division,record,last_fight
0,1,Valentina Shevchenko,1867,1871,4,10.8,flyweight,15-3-1,2025-11-15
1,2,Erin Blanchfield,1846,1850,4,10.8,flyweight,8-1,2025-11-15
2,3,Alexa Grasso,1844,1844,0,0.9,flyweight,10-5-1,2026-09-12
3,4,Tatiana Suarez,1842,1842,0,6.0,strawweight,9-1,2026-04-11
4,5,Zhang Weili,1825,1829,4,10.8,flyweight,10-3,2025-11-15
5,6,Manon Fiorot,1805,1805,0,0.9,flyweight,8-2,2026-09-12
6,7,Mackenzie Dern,1785,1785,0,1.8,strawweight,12-5,2026-08-15
7,8,Natalia Silva,1784,1784,0,0.2,flyweight,9-0,2026-10-03
8,9,Virna Jandiroba,1771,1771,0,6.2,strawweight,9-4,2026-04-04
9,10,Denise Gomes,1767,1767,0,1.4,strawweight,7-2,2026-08-29
